# Candle Backfill v3 — Maximum-Depth Backfill

This notebook backfills all (connector, pair, interval) combos in your MongoDB
`candles` collection to the maximum depth available from each exchange, up to
a 10-year ceiling.

**Key differences from v2:**

1. **Config-driven.** Exchange URLs, interval targets, and rate-limit pacing
   live in `candle_backfill_config.yaml` next to this notebook. To add a new
   exchange (e.g., Kraken), edit the config and implement one fetcher +
   ingester function in Cell 4/5.

2. **Auto-probed history floors.** Before running, v3 probes each
   `(exchange, interval)` pair to discover the earliest timestamp the exchange
   will serve, and caches the result in `candle_coverage_probes`. No wasted
   requests reaching for data that doesn't exist.

3. **Per-interval target depth.** Each interval has its own target depth in
   the config (e.g., 1m → 60 days, 1d → 10 years). The effective backfill
   window is `min(target_days, exchange_floor)`.

4. **Parallel across exchanges, serial within.** Exchanges run in parallel
   threads (Binance + MEXC + NonKYC + Coinbase all at once). Within each
   exchange, pairs run serially to avoid triggering IP bans.

5. **Preview mode.** Cell 8 prints estimated request counts and wall-clock
   time per exchange before you commit to a run.

**Prerequisites:**
```bash
pip install pymongo requests pyyaml
```

**Place `candle_backfill_config.yaml` in the same directory as this notebook.**


In [1]:
# ── Cell 1: Load configuration ───────────────────────────────────────────────
import os
import pathlib
import yaml

CONFIG_PATH = pathlib.Path(os.environ.get(
    "CANDLE_BACKFILL_CONFIG",
    str(pathlib.Path.cwd() / "candle_backfill_config.yaml"),
))

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Config file not found at {CONFIG_PATH}. "
        f"Set CANDLE_BACKFILL_CONFIG env var or place the yaml next to this notebook."
    )

with open(CONFIG_PATH) as f:
    CFG = yaml.safe_load(f)

# Mongo connection (env-var aware, same pattern as v2)
_m = CFG["mongo"]
TRUENAS_IP   = os.environ.get(_m["truenas_ip_env_var"], _m["default_truenas_ip"])
TRUENAS_PASS = os.environ.get(_m["truenas_pass_env_var"], _m["default_truenas_pass"])
MONGO_URI = os.environ.get(
    _m["uri_env_var"],
    f"mongodb://admin:{TRUENAS_PASS}@{TRUENAS_IP}:27017/{_m['database']}?authSource=admin",
)
MONGO_DATABASE     = _m["database"]
COLLECTION_NAME    = _m["collection"]
PROBE_COLL_NAME    = _m["probe_cache_collection"]
PAIR_START_COLL_NAME = f"{PROBE_COLL_NAME}_pair_starts"
COVERAGE_COLL_NAME   = f"{COLLECTION_NAME}_coverage"

# HTTP tuning
HTTP_TIMEOUT       = CFG["http"]["timeout_seconds"]
HTTP_MAX_RETRIES   = CFG["http"]["max_retries"]
HTTP_RETRY_BACKOFF = CFG["http"]["retry_backoff"]

# Per-interval target depth (days); capped by absolute_max_days
INTERVAL_TARGETS_DAYS = CFG["interval_targets_days"]
ABSOLUTE_MAX_DAYS     = CFG["absolute_max_days"]
RECENT_DAYS_OWNED = int(CFG.get("recent_days_owned_by_other_process", 0))

# Exchange table (URL, pacing, interval-map, max_per_request)
EXCHANGES = CFG["exchanges"]

SCHEMA_VERSION = 3

# Interval → seconds. Used everywhere.
INTERVAL_SECONDS = {
    "1m":  60,
    "3m":  180,
    "5m":  300,
    "15m": 900,
    "30m": 1800,
    "1h":  3600,
    "4h":  14400,
    "8h":  28800,
    "12h": 43200,
    "1d":  86400,
}

print(f"Loaded config from: {CONFIG_PATH}")
print(f"Configured exchanges: {', '.join(sorted(EXCHANGES.keys()))}")
print(f"Configured intervals: {', '.join(sorted(INTERVAL_TARGETS_DAYS.keys(), key=lambda k: INTERVAL_SECONDS.get(k, 0)))}")
print(f"Absolute max lookback: {ABSOLUTE_MAX_DAYS} days ({ABSOLUTE_MAX_DAYS / 365:.1f} years)")
print(f"Pair-start cache collection: {PAIR_START_COLL_NAME}")
print(f"Coverage collection: {COVERAGE_COLL_NAME}")
print(f"Skipping last {RECENT_DAYS_OWNED} days for existing pairs (0 = disabled)")


Loaded config from: /quants-lab/research_notebooks/market_lab/pmm_dynamic/notebooks/mongo_tools/candle_backfill_config.yaml
Configured exchanges: binance, coinbase, kraken, mexc, nonkyc
Configured intervals: 1m, 3m, 5m, 15m, 30m, 1h, 4h, 8h, 12h, 1d
Absolute max lookback: 3650 days (10.0 years)
Pair-start cache collection: candle_coverage_probes_pair_starts
Coverage collection: candles_coverage
Skipping last 7 days for existing pairs (0 = disabled)


In [2]:
# ── Cell 2: Imports & helpers ────────────────────────────────────────────────
import math
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from typing import Any, Optional

import requests
from pymongo import MongoClient, UpdateOne
from pymongo.errors import BulkWriteError


def utc_now_ts() -> int:
    return int(time.time())


def align_floor(ts: int, step: int) -> int:
    return (ts // step) * step if step > 0 else ts


def last_closed_open_ts(now_ts: int, step: int) -> int:
    return align_floor(now_ts, step) - step


def fmt_ts(ts: int) -> str:
    return datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M")


def ts_to_iso8601(ts: int) -> str:
    return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat().replace("+00:00", "Z")


def safe_float(x) -> float:
    try:
        return float(x)
    except Exception:
        return float("nan")


def to_hbot_pair(pair: str) -> str:
    p = pair.strip().upper()
    if "/" in p:
        base, quote = p.split("/", 1)
        return f"{base}-{quote}"
    if "_" in p:
        base, quote = p.split("_", 1)
        return f"{base}-{quote}"
    return p.replace("/", "-").replace("_", "-")


def split_hbot_pair(hbot_pair: str):
    return hbot_pair.split("-", 1)


def to_nonkyc_symbol(hbot_pair: str) -> str:
    return hbot_pair.replace("-", "_")


def to_binance_symbol(hbot_pair: str) -> str:
    return hbot_pair.replace("-", "")


def to_mexc_symbol(hbot_pair: str) -> str:
    return hbot_pair.replace("-", "")


def to_coinbase_product_id(hbot_pair: str) -> str:
    return hbot_pair  # already BASE-QUOTE


# Kraken uses "XBT" instead of "BTC", and "XDG" instead of "DOGE", in some
# contexts. This translation is applied to the request symbol; the response
# key returned by Kraken is NOT guaranteed to match and must be discovered
# at parse time (see fetch_kraken_candles).
_KRAKEN_BASE_TRANSLATIONS = {
    "BTC": "XBT",
    "DOGE": "XDG",
}


def to_kraken_symbol(hbot_pair: str) -> str:
    base, quote = split_hbot_pair(hbot_pair)
    kb = _KRAKEN_BASE_TRANSLATIONS.get(base, base)
    return f"{kb}{quote}"


def compute_qc_flags(o, h, l, c, v):
    flags = []
    if h < max(o, c): flags.append("high_lt_open_close")
    if l > min(o, c): flags.append("low_gt_open_close")
    if h < l: flags.append("high_lt_low")
    if v < 0: flags.append("neg_volume")
    for name, val in [("open", o), ("high", h), ("low", l), ("close", c), ("volume", v)]:
        if val != val: flags.append(f"nan_{name}")
    return flags


# Connection pooling for high-concurrency HTTP
from requests.adapters import HTTPAdapter

session = requests.Session()
_adapter = HTTPAdapter(pool_connections=16, pool_maxsize=32, max_retries=0)
session.mount("https://", _adapter)
session.mount("http://", _adapter)


class _TokenBucket:
    """Thread-safe token bucket with auto-decay on rate-limit signal.

    rate_per_sec   steady-state refill rate (tokens/s)
    burst_capacity max tokens in bucket (for burst tolerance)
    """

    def __init__(self, rate_per_sec: float, burst_capacity: float):
        self.rate = float(rate_per_sec)
        self.capacity = float(burst_capacity)
        self.tokens = float(burst_capacity)
        self.last = time.monotonic()
        self._cv = threading.Condition()
        self._min_rate = 0.5

    def acquire(self) -> None:
        with self._cv:
            while True:
                now = time.monotonic()
                elapsed = now - self.last
                self.last = now
                self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
                if self.tokens >= 1.0:
                    self.tokens -= 1.0
                    return
                wait = (1.0 - self.tokens) / self.rate
                self._cv.wait(timeout=wait)

    def on_rate_limited(self, retry_after_seconds):
        with self._cv:
            self.rate = max(self._min_rate, self.rate * 0.5)
            self.tokens = 0.0
            self._cv.notify_all()
        if retry_after_seconds:
            time.sleep(retry_after_seconds)

    def stats(self):
        with self._cv:
            return {"rate_per_sec": self.rate, "tokens": self.tokens,
                    "capacity": self.capacity}


# Per-exchange tuning. These values have been empirically validated for NonKYC.
# For other exchanges, values are conservative starting points close to the
# previous effective rates. If you need to reduce load, lower rate_per_sec.
_EXCHANGE_TUNING = {
    "nonkyc":   {"rate_per_sec": 4.0, "burst": 8.0, "parallel": 8},
    "binance":  {"rate_per_sec": 8.0, "burst": 16.0, "parallel": 2},
    "mexc":     {"rate_per_sec": 5.0,  "burst": 10.0, "parallel": 2},
    "coinbase": {"rate_per_sec": 3.0,  "burst": 6.0,  "parallel": 2},
    "kraken":   {"rate_per_sec": 1.0, "burst": 2.0,  "parallel": 1},
}

_exchange_buckets: dict = {}
_buckets_mutex = threading.Lock()


def _get_exchange_bucket(exchange: str) -> _TokenBucket:
    with _buckets_mutex:
        b = _exchange_buckets.get(exchange)
        if b is None:
            cfg = _EXCHANGE_TUNING.get(exchange, {"rate_per_sec": 2.0, "burst": 4.0})
            b = _TokenBucket(cfg["rate_per_sec"], cfg["burst"])
            _exchange_buckets[exchange] = b
        return b


def get_exchange_parallel_workers(exchange: str) -> int:
    return _EXCHANGE_TUNING.get(exchange, {}).get("parallel", 1)


def http_get_json(url: str, params: dict = None, prefix: str = "",
                  exchange: Optional[str] = None,
                  headers: Optional[dict] = None) -> Any:
    """GET + JSON with per-request session, watchdog, and robust 5xx retries.

    Behavior:
    - Watchdog thread enforces HARD_DEADLINE_SECONDS wall-clock ceiling.
    - 5xx errors (500, 502, 503, 504) are retried up to RETRY_BUDGET_5XX
      times with exponential backoff + random jitter. These are treated
      as transient backend hiccups.
    - Non-5xx errors (connection errors, timeouts, other exceptions)
      retry up to HTTP_MAX_RETRIES times using the standard backoff.
    - Rate limits (429, 418, 1015) use the token bucket's auto-decay.
    - Each attempt gets a fresh requests.Session so closing it on
      timeout doesn't affect other attempts.
    """
    import random  # stdlib

    HARD_DEADLINE_SECONDS = 20.0
    RETRY_BUDGET_5XX = 6            # More aggressive than HTTP_MAX_RETRIES for 5xx
    MAX_BACKOFF_5XX = 30.0          # Cap backoff so a single retry doesn't sleep forever

    bucket = _get_exchange_bucket(exchange) if exchange else None
    last_err = None
    attempt = 0
    attempt_5xx = 0

    # Use a looser attempt cap that accommodates either budget.
    max_attempts = max(HTTP_MAX_RETRIES, RETRY_BUDGET_5XX)

    while attempt < max_attempts:
        attempt += 1
        if bucket is not None:
            bucket.acquire()

        local_session = requests.Session()
        adapter = requests.adapters.HTTPAdapter(
            pool_connections=1, pool_maxsize=1, max_retries=0,
        )
        local_session.mount("https://", adapter)
        local_session.mount("http://", adapter)

        done = threading.Event()

        def _watchdog():
            if not done.wait(HARD_DEADLINE_SECONDS):
                try:
                    local_session.close()
                    print(f"  {prefix}Watchdog: killed hung connection after "
                          f"{HARD_DEADLINE_SECONDS:.0f}s")
                except Exception:
                    pass

        watchdog = threading.Thread(target=_watchdog, daemon=True)
        watchdog.start()

        try:
            resp = local_session.get(
                url, params=params, headers=headers,
                timeout=(5, 15),
            )
            if resp.status_code in (429, 418, 1015):
                retry_after = resp.headers.get("Retry-After")
                sleep_s = float(retry_after) if retry_after else HTTP_RETRY_BACKOFF ** attempt
                print(f"  {prefix}Rate limited (HTTP {resp.status_code}), "
                      f"sleeping {sleep_s:.1f}s and halving rate")
                if bucket is not None:
                    bucket.on_rate_limited(sleep_s)
                else:
                    time.sleep(sleep_s)
                done.set()
                continue
            if resp.status_code in (500, 502, 503, 504):
                # Transient backend error — treat with the 5xx-specific budget
                attempt_5xx += 1
                last_err = RuntimeError(f"HTTP {resp.status_code}: {resp.text[:200]}")
                done.set()
                if attempt_5xx >= RETRY_BUDGET_5XX:
                    break
                # Exponential backoff with jitter (0.7x - 1.3x multiplier)
                raw_backoff = min(MAX_BACKOFF_5XX,
                                   HTTP_RETRY_BACKOFF ** attempt_5xx)
                sleep_s = raw_backoff * (0.7 + 0.6 * random.random())
                print(f"  {prefix}HTTP {resp.status_code} transient, retrying in {sleep_s:.1f}s "
                      f"(attempt {attempt_5xx}/{RETRY_BUDGET_5XX})")
                time.sleep(sleep_s)
                continue
            resp.raise_for_status()
            result = resp.json()
            done.set()
            return result
        except (requests.exceptions.ReadTimeout,
                requests.exceptions.ConnectionError) as e:
            last_err = e
            done.set()
            if attempt >= HTTP_MAX_RETRIES:
                break
            time.sleep(HTTP_RETRY_BACKOFF ** (attempt - 1))
        except Exception as e:
            last_err = e
            done.set()
            if attempt >= HTTP_MAX_RETRIES:
                break
            time.sleep(HTTP_RETRY_BACKOFF ** (attempt - 1))
        finally:
            try:
                local_session.close()
            except Exception:
                pass

    raise last_err


def normalize_candle_doc(
    connector, hbot_pair, interval, ts_open, open_, high, low, close,
    base_volume, interval_seconds, now_ts,
    quote_volume=None, trade_count=None,
    taker_buy_base_volume=None, taker_buy_quote_volume=None,
):
    base, quote = split_hbot_pair(hbot_pair)
    close_ts = ts_open + interval_seconds
    is_closed = close_ts <= now_ts
    qv_estimated = quote_volume is None
    if quote_volume is None:
        vwap = (high + low + close) / 3.0
        quote_volume = base_volume * vwap
    qc_flags = compute_qc_flags(open_, high, low, close, base_volume)
    doc = {
        "schema_version": SCHEMA_VERSION,
        "connector": connector,
        "trading_pair": hbot_pair,
        "interval": interval,
        "timestamp": ts_open,
        "open_ts": ts_open,
        "close_ts": close_ts,
        "is_closed": is_closed,
        "base_asset": base,
        "quote_asset": quote,
        "open": float(open_),
        "high": float(high),
        "low": float(low),
        "close": float(close),
        "volume": float(base_volume),
        "base_volume": float(base_volume),
        "quote_volume": float(quote_volume),
        "quote_volume_is_estimated": qv_estimated,
        "qc_ok": len(qc_flags) == 0,
        "qc_flags": qc_flags,
        "is_synthetic": False,
        "updated_at": now_ts,
    }
    if trade_count is not None: doc["trade_count"] = int(trade_count)
    if taker_buy_base_volume is not None: doc["taker_buy_base_volume"] = float(taker_buy_base_volume)
    if taker_buy_quote_volume is not None: doc["taker_buy_quote_volume"] = float(taker_buy_quote_volume)
    return doc


def upsert_candles(coll, candles, is_backfill=False):
    if not candles:
        return 0
    now_ts = utc_now_ts()
    if is_backfill:
        for c in candles:
            c.setdefault("ingested_at", now_ts)
        try:
            res = coll.insert_many(candles, ordered=False)
            return len(res.inserted_ids)
        except BulkWriteError as bwe:
            return int(bwe.details.get("nInserted", 0))
    ops = []
    for c in candles:
        key = {"connector": c["connector"], "trading_pair": c["trading_pair"],
               "interval": c["interval"], "timestamp": c["timestamp"]}
        ops.append(UpdateOne(key, {"$set": c, "$setOnInsert": {"ingested_at": now_ts}}, upsert=True))
    try:
        res = coll.bulk_write(ops, ordered=False)
        return int(res.upserted_count + res.modified_count)
    except BulkWriteError:
        return 0


def merge_step_ranges(ranges, step):
    """Merge overlapping/adjacent [start_ts, end_ts] ranges in candle-open space."""
    cleaned = [(int(s), int(e)) for s, e in ranges if s is not None and e is not None and s <= e]
    if not cleaned:
        return []
    cleaned.sort(key=lambda x: (x[0], x[1]))
    merged = [cleaned[0]]
    for s, e in cleaned[1:]:
        last_s, last_e = merged[-1]
        if s <= last_e + step:
            merged[-1] = (last_s, max(last_e, e))
        else:
            merged.append((s, e))
    return merged


def format_ts_ranges(ranges, limit=5):
    if not ranges:
        return "—"
    parts = []
    for s, e in ranges[:limit]:
        parts.append(f"{fmt_ts(s)} → {fmt_ts(e)}")
    if len(ranges) > limit:
        parts.append(f"... (+{len(ranges) - limit} more)")
    return " | ".join(parts)


def scan_doc_ranges(coll, connector, pair, interval, step, window_start, window_end):
    """Return contiguous candle ranges already present in Mongo for one series."""
    effective_start = align_floor(window_start, step)
    effective_end = align_floor(window_end, step)
    if effective_end < effective_start:
        return {"ranges": [], "count": 0, "first_ts": None, "last_ts": None}

    cursor = coll.find(
        {
            "connector": connector,
            "trading_pair": pair,
            "interval": interval,
            "timestamp": {"$gte": effective_start, "$lte": effective_end},
        },
        projection={"timestamp": 1, "_id": 0},
        sort=[("timestamp", 1)],
        batch_size=5000,
    )

    ranges = []
    count = 0
    first_ts = None
    last_ts = None
    range_start = None
    prev = None

    for doc in cursor:
        ts = int(doc["timestamp"])
        count += 1
        if first_ts is None:
            first_ts = ts
            range_start = ts
        elif prev is not None and ts > prev + step:
            ranges.append((range_start, prev))
            range_start = ts
        prev = ts
        last_ts = ts

    if range_start is not None and prev is not None:
        ranges.append((range_start, prev))

    return {
        "ranges": ranges,
        "count": count,
        "first_ts": first_ts,
        "last_ts": last_ts,
    }


def load_recorded_coverage_ranges(coverage_coll, connector, pair, interval, step, window_start, window_end):
    """Load recorded backfill coverage receipts for one series, merged/clamped."""
    effective_start = align_floor(window_start, step)
    effective_end = align_floor(window_end, step)
    if effective_end < effective_start:
        return []

    cursor = coverage_coll.find(
        {
            "connector": connector,
            "trading_pair": pair,
            "interval": interval,
            "end_ts": {"$gte": effective_start},
            "start_ts": {"$lte": effective_end},
        },
        projection={"start_ts": 1, "end_ts": 1, "_id": 0},
        sort=[("start_ts", 1), ("end_ts", 1)],
    )

    ranges = []
    for doc in cursor:
        s = max(effective_start, int(doc["start_ts"]))
        e = min(effective_end, int(doc["end_ts"]))
        if s <= e:
            ranges.append((s, e))
    return merge_step_ranges(ranges, step)


def complement_ranges(window_start, window_end, covered_ranges, step):
    """Return uncovered ranges inside [window_start, window_end] inclusive."""
    effective_start = align_floor(window_start, step)
    effective_end = align_floor(window_end, step)
    if effective_end < effective_start:
        return []

    covered = merge_step_ranges(covered_ranges, step)
    if not covered:
        return [(effective_start, effective_end)]

    gaps = []
    cursor = effective_start
    for s, e in covered:
        if e < effective_start or s > effective_end:
            continue
        s = max(s, effective_start)
        e = min(e, effective_end)
        if cursor < s:
            gaps.append((cursor, s - step))
        cursor = max(cursor, e + step)
    if cursor <= effective_end:
        gaps.append((cursor, effective_end))
    return [(s, e) for s, e in gaps if s <= e]


def summarize_series_window(coll, connector, pair, interval, step, window_start, window_end):
    """Summarize doc coverage + uncovered ranges for one series window.

    For Kraken, coverage receipts are unioned with observed candles so that
    intervals with no trades (and therefore no candles) do not get retried
    forever once a range has been scanned successfully.
    """
    effective_start = align_floor(window_start, step)
    effective_end = align_floor(window_end, step)
    expected = ((effective_end - effective_start) // step) + 1 if effective_end >= effective_start else 0

    doc_summary = scan_doc_ranges(coll, connector, pair, interval, step, effective_start, effective_end)
    recorded_ranges = []
    if connector == "kraken":
        recorded_ranges = load_recorded_coverage_ranges(
            coverage_coll, connector, pair, interval, step, effective_start, effective_end
        )

    covered_ranges = list(doc_summary["ranges"])
    if connector == "kraken" and recorded_ranges:
        covered_ranges = merge_step_ranges(doc_summary["ranges"] + recorded_ranges, step)

    gap_ranges = complement_ranges(effective_start, effective_end, covered_ranges, step)
    missing = sum(((e - s) // step) + 1 for s, e in gap_ranges)
    covered_buckets = max(0, expected - missing)
    pct = (covered_buckets / expected * 100.0) if expected > 0 else 100.0

    return {
        "effective_start": effective_start,
        "effective_end": effective_end,
        "expected": expected,
        "docs_in_window": doc_summary["count"],
        "first_doc_ts": doc_summary["first_ts"],
        "last_doc_ts": doc_summary["last_ts"],
        "doc_ranges": doc_summary["ranges"],
        "recorded_ranges": recorded_ranges,
        "covered_ranges": covered_ranges,
        "gap_ranges": gap_ranges,
        "covered_buckets": covered_buckets,
        "missing": missing,
        "pct": pct,
    }


def record_coverage_range(coverage_coll, connector, pair, interval, start_ts, end_ts, source="backfill"):
    if end_ts < start_ts:
        return
    now_ts = utc_now_ts()
    coverage_coll.update_one(
        {
            "connector": connector,
            "trading_pair": pair,
            "interval": interval,
            "start_ts": int(start_ts),
            "end_ts": int(end_ts),
        },
        {
            "$set": {
                "connector": connector,
                "trading_pair": pair,
                "interval": interval,
                "start_ts": int(start_ts),
                "end_ts": int(end_ts),
                "source": source,
                "updated_at": now_ts,
            },
            "$setOnInsert": {"created_at": now_ts},
        },
        upsert=True,
    )


print("Helpers loaded.")


Helpers loaded.


In [3]:
# ── Cell 3: Exchange-specific candle fetchers ────────────────────────────────
#
# Each fetcher takes (hbot_pair, interval, window params) and returns a list of
# candles. To add a new exchange, add a fetcher here matching this signature
# style, then add a corresponding ingester in Cell 4.

def fetch_nonkyc_candles(hbot_pair, interval, to_ts, count):
    ex_cfg = EXCHANGES["nonkyc"]
    res = ex_cfg["interval_map"].get(interval)
    if res is None:
        return []
    url = ex_cfg["base_url"].rstrip("/") + "/market/candles"
    params = {
        "symbol": to_nonkyc_symbol(hbot_pair),
        "resolution": int(res),
        "countBack": int(count),
        "to": int(to_ts),
    }
    data = http_get_json(url, params, prefix="[nonkyc] ",
                         exchange="nonkyc",
                         headers={"Connection": "close"})
    if not isinstance(data, dict):
        return []
    candles = []
    # Format 1: TradingView UDF arrays
    if data.get("s") == "ok" and "t" in data:
        t_arr = data.get("t", [])
        o_arr = data.get("o", []); h_arr = data.get("h", [])
        l_arr = data.get("l", []); c_arr = data.get("c", [])
        v_arr = data.get("v", [])
        for i in range(min(len(t_arr), len(o_arr), len(h_arr), len(l_arr), len(c_arr), len(v_arr))):
            candles.append({
                "timestamp": int(int(t_arr[i]) // 1000),
                "open": safe_float(o_arr[i]), "high": safe_float(h_arr[i]),
                "low": safe_float(l_arr[i]), "close": safe_float(c_arr[i]),
                "volume": safe_float(v_arr[i]),
            })
        return candles
    # Format 2: { bars: [...] }
    bars = data.get("bars")
    if isinstance(bars, list):
        for bar in bars:
            if not isinstance(bar, dict): continue
            ts_val = int(bar.get("time", 0))
            if ts_val > 1e12: ts_val = ts_val // 1000
            candles.append({
                "timestamp": ts_val,
                "open": safe_float(bar.get("open")), "high": safe_float(bar.get("high")),
                "low": safe_float(bar.get("low")), "close": safe_float(bar.get("close")),
                "volume": safe_float(bar.get("volume")),
            })
    return candles


def fetch_binance_candles(hbot_pair, interval, start_ms, end_ms, limit):
    ex_cfg = EXCHANGES["binance"]
    api_interval = ex_cfg["interval_map"].get(interval)
    if api_interval is None:
        return []
    url = ex_cfg["base_url"].rstrip("/") + "/api/v3/klines"
    params = {"symbol": to_binance_symbol(hbot_pair), "interval": api_interval,
              "startTime": int(start_ms), "endTime": int(end_ms), "limit": int(limit)}
    data = http_get_json(url, params, prefix="[binance] ", exchange="binance")
    return data if isinstance(data, list) else []


def fetch_mexc_candles(hbot_pair, interval, start_ms, end_ms, limit):
    ex_cfg = EXCHANGES["mexc"]
    api_interval = ex_cfg["interval_map"].get(interval)
    if api_interval is None:
        return []
    url = ex_cfg["base_url"].rstrip("/") + "/api/v3/klines"
    params = {"symbol": to_mexc_symbol(hbot_pair), "interval": api_interval,
              "startTime": int(start_ms), "endTime": int(end_ms), "limit": int(limit)}
    data = http_get_json(url, params, prefix="[mexc] ", exchange="mexc")
    return data if isinstance(data, list) else []


def fetch_coinbase_candles(hbot_pair, interval, start_ts, end_ts):
    ex_cfg = EXCHANGES["coinbase"]
    granularity = ex_cfg["interval_map"].get(interval)
    if granularity is None:
        return []
    product_id = to_coinbase_product_id(hbot_pair)
    url = ex_cfg["base_url"].rstrip("/") + f"/products/{product_id}/candles"
    params = {"granularity": int(granularity),
              "start": ts_to_iso8601(int(start_ts)), "end": ts_to_iso8601(int(end_ts))}
    data = http_get_json(url, params, prefix="[coinbase] ", exchange="coinbase")
    return data if isinstance(data, list) else []


# --- Adding a new exchange? Template: ---

def fetch_kraken_candles(hbot_pair, interval, since_ts=None):
    """Fetch OHLC candles from Kraken's public /0/public/OHLC endpoint.

    Kraken caps this endpoint at 720 bars per response and does NOT support
    paging further back. Deep history is therefore handled in Cell 4 via the
    /0/public/Trades endpoint plus local candle aggregation.
    """
    ex_cfg = EXCHANGES["kraken"]
    api_interval = ex_cfg["interval_map"].get(interval)
    if api_interval is None:
        return []
    url = ex_cfg["base_url"].rstrip("/") + "/0/public/OHLC"
    params = {
        "pair": to_kraken_symbol(hbot_pair),
        "interval": int(api_interval),
    }
    if since_ts is not None:
        params["since"] = int(since_ts)
    data = http_get_json(url, params, prefix="[kraken] ", exchange="kraken")

    # Kraken response shape:
    #   {"error": [...], "result": {"<pair_key>": [[ts,o,h,l,c,vwap,vol,count], ...], "last": <ts>}}
    if not isinstance(data, dict):
        return []
    if data.get("error"):
        print(f"  [kraken] API error for {hbot_pair}: {data['error']}")
        return []
    result = data.get("result", {})
    if not isinstance(result, dict):
        return []
    pair_key = None
    for k in result.keys():
        if k != "last":
            pair_key = k
            break
    if pair_key is None:
        return []
    rows = result.get(pair_key, [])
    if not isinstance(rows, list):
        return []

    candles = []
    for r in rows:
        if not isinstance(r, (list, tuple)) or len(r) < 7:
            continue
        ts_open = int(r[0])
        candles.append({
            "timestamp": ts_open,
            "open": safe_float(r[1]),
            "high": safe_float(r[2]),
            "low": safe_float(r[3]),
            "close": safe_float(r[4]),
            "volume": safe_float(r[6]),
        })
    return candles


def fetch_kraken_trades(hbot_pair, since_ns=None):
    """Fetch historical trades from Kraken's public /0/public/Trades endpoint.

    Returns
    -------
    tuple[list[dict], str|int|None]
        (trades, next_cursor)

    Each trade dict has:
      {price, volume, timestamp}

    Notes
    -----
    - `since_ns=0` starts at the beginning of the market for that pair.
    - Subsequent requests should pass the `last` cursor returned by the
      previous response.
    """
    ex_cfg = EXCHANGES["kraken"]
    url = ex_cfg["base_url"].rstrip("/") + "/0/public/Trades"
    params = {"pair": to_kraken_symbol(hbot_pair)}
    if since_ns is not None:
        params["since"] = str(since_ns)

    data = http_get_json(url, params, prefix="[kraken] ", exchange="kraken")
    if not isinstance(data, dict):
        return [], None
    if data.get("error"):
        print(f"  [kraken] Trades API error for {hbot_pair}: {data['error']}")
        return [], None

    result = data.get("result", {})
    if not isinstance(result, dict):
        return [], None

    pair_key = None
    for k in result.keys():
        if k != "last":
            pair_key = k
            break
    rows = result.get(pair_key, []) if pair_key is not None else []
    if not isinstance(rows, list):
        rows = []

    trades = []
    for r in rows:
        if not isinstance(r, (list, tuple)) or len(r) < 3:
            continue
        price = safe_float(r[0])
        volume = safe_float(r[1])
        ts = float(r[2])
        if price != price or volume != volume:
            continue
        trades.append({
            "price": price,
            "volume": volume,
            "timestamp": ts,
        })
    return trades, result.get("last")


FETCHERS = {
    "nonkyc": fetch_nonkyc_candles,
    "binance": fetch_binance_candles,
    "mexc": fetch_mexc_candles,
    "coinbase": fetch_coinbase_candles,
    "kraken": fetch_kraken_candles,
}

print(f"Fetchers loaded: {', '.join(sorted(FETCHERS.keys()))}")


Fetchers loaded: binance, coinbase, kraken, mexc, nonkyc


In [4]:
# ── Cell 4: Range ingesters (per exchange) ──────────────────────────────────
#
# Each ingester paginates a time range and writes normalized candle docs to Mongo.

def ingest_nonkyc_range(coll, hbot_pair, interval, start_ts, end_ts):
    ex_cfg = EXCHANGES["nonkyc"]
    if ex_cfg["interval_map"].get(interval) is None:
        return 0
    step = INTERVAL_SECONDS[interval]
    now_ts = utc_now_ts()
    max_per = ex_cfg["max_per_request"]
    page_to = end_ts + step - 1
    total = 0
    safety = 0
    started = time.perf_counter()
    last_heartbeat = started
    requests_made = 0
    while page_to >= start_ts and safety < 100000:
        safety += 1
        requests_made += 1
        bars = fetch_nonkyc_candles(hbot_pair, interval, to_ts=page_to, count=max_per)
        # Heartbeat: show live progress when a single pair is taking a while
        now_pc = time.perf_counter()
        if now_pc - last_heartbeat >= 30.0:
            elapsed = now_pc - started
            print(f"    [nonkyc] {hbot_pair} {interval}: "
                  f"{requests_made} reqs, {total:,} written, {elapsed:.0f}s elapsed")
            last_heartbeat = now_pc
        if not bars:
            break
        docs = []
        oldest = None
        for b in bars:
            ts_open = int(b["timestamp"])
            if ts_open < start_ts or ts_open > end_ts:
                continue
            oldest = ts_open if oldest is None else min(oldest, ts_open)
            docs.append(normalize_candle_doc(
                connector="nonkyc", hbot_pair=hbot_pair, interval=interval,
                ts_open=ts_open, open_=b["open"], high=b["high"], low=b["low"],
                close=b["close"], base_volume=b["volume"],
                interval_seconds=step, now_ts=now_ts,
            ))
        if docs:
            total += upsert_candles(coll, docs, is_backfill=True)
        if oldest is None or oldest <= start_ts or oldest >= page_to:
            break
        page_to = oldest - 1
        # No explicit sleep — the token bucket inside http_get_json paces requests
    return total


def ingest_binance_range(coll, hbot_pair, interval, start_ts, end_ts):
    ex_cfg = EXCHANGES["binance"]
    if ex_cfg["interval_map"].get(interval) is None:
        return 0
    step = INTERVAL_SECONDS[interval]
    now_ts = utc_now_ts()
    max_per = ex_cfg["max_per_request"]
    delay = ex_cfg["request_delay_seconds"]
    cur = start_ts
    total = 0
    consecutive_empty = 0
    safety = 0
    while cur <= end_ts and safety < 100000:
        safety += 1
        window_end = min(cur + (max_per - 1) * step, end_ts)
        rows = fetch_binance_candles(hbot_pair, interval, cur * 1000, window_end * 1000, max_per)
        if not rows:
            consecutive_empty += 1
            if consecutive_empty >= 3:
                # Three empty windows in a row — the pair likely didn't exist yet.
                break
            cur = window_end + step
            time.sleep(delay)
            continue
        consecutive_empty = 0
        docs = []
        max_row_ts = None
        for r in rows:
            ts_open = int(r[0]) // 1000
            if ts_open < start_ts or ts_open > end_ts:
                continue
            max_row_ts = ts_open if max_row_ts is None else max(max_row_ts, ts_open)
            docs.append(normalize_candle_doc(
                connector="binance", hbot_pair=hbot_pair, interval=interval,
                ts_open=ts_open, open_=safe_float(r[1]), high=safe_float(r[2]),
                low=safe_float(r[3]), close=safe_float(r[4]),
                base_volume=safe_float(r[5]), interval_seconds=step, now_ts=now_ts,
                quote_volume=safe_float(r[7]),
                trade_count=int(r[8]) if len(r) > 8 else None,
                taker_buy_base_volume=safe_float(r[9]) if len(r) > 9 else None,
                taker_buy_quote_volume=safe_float(r[10]) if len(r) > 10 else None,
            ))
        if docs:
            total += upsert_candles(coll, docs, is_backfill=True)
        cur = (max_row_ts + step) if max_row_ts else (cur + max_per * step)
        time.sleep(delay)
    return total


def ingest_mexc_range(coll, hbot_pair, interval, start_ts, end_ts):
    ex_cfg = EXCHANGES["mexc"]
    if ex_cfg["interval_map"].get(interval) is None:
        return 0
    step = INTERVAL_SECONDS[interval]
    now_ts = utc_now_ts()
    max_per = ex_cfg["max_per_request"]
    delay = ex_cfg["request_delay_seconds"]
    cur = start_ts
    total = 0
    consecutive_empty = 0
    safety = 0
    while cur <= end_ts and safety < 100000:
        safety += 1
        window_end = min(cur + (max_per - 1) * step, end_ts)
        rows = fetch_mexc_candles(hbot_pair, interval, cur * 1000, window_end * 1000, max_per)
        if not rows:
            consecutive_empty += 1
            if consecutive_empty >= 3:
                break
            cur = window_end + step
            time.sleep(delay)
            continue
        consecutive_empty = 0
        docs = []
        max_row_ts = None
        for r in rows:
            ts_open = int(r[0]) // 1000
            if ts_open < start_ts or ts_open > end_ts:
                continue
            max_row_ts = ts_open if max_row_ts is None else max(max_row_ts, ts_open)
            docs.append(normalize_candle_doc(
                connector="mexc", hbot_pair=hbot_pair, interval=interval,
                ts_open=ts_open, open_=safe_float(r[1]), high=safe_float(r[2]),
                low=safe_float(r[3]), close=safe_float(r[4]),
                base_volume=safe_float(r[5]), interval_seconds=step, now_ts=now_ts,
                quote_volume=safe_float(r[7]) if len(r) > 7 else None,
            ))
        if docs:
            total += upsert_candles(coll, docs, is_backfill=True)
        cur = (max_row_ts + step) if max_row_ts else (cur + max_per * step)
        time.sleep(delay)
    return total


def ingest_coinbase_range(coll, hbot_pair, interval, start_ts, end_ts):
    ex_cfg = EXCHANGES["coinbase"]
    if ex_cfg["interval_map"].get(interval) is None:
        return 0
    step = INTERVAL_SECONDS[interval]
    now_ts = utc_now_ts()
    max_per = ex_cfg["max_per_request"]
    delay = ex_cfg["request_delay_seconds"]
    cur = start_ts
    total = 0
    consecutive_empty = 0
    safety = 0
    while cur <= end_ts and safety < 100000:
        safety += 1
        window_end = min(cur + (max_per - 1) * step, end_ts)
        rows = fetch_coinbase_candles(hbot_pair, interval, cur, window_end)
        if not rows:
            consecutive_empty += 1
            if consecutive_empty >= 3:
                break
            cur = window_end + step
            time.sleep(delay)
            continue
        consecutive_empty = 0
        docs = []
        max_row_ts = None
        for r in rows:
            ts_open = int(r[0])
            if ts_open < start_ts or ts_open > end_ts:
                continue
            max_row_ts = ts_open if max_row_ts is None else max(max_row_ts, ts_open)
            docs.append(normalize_candle_doc(
                connector="coinbase", hbot_pair=hbot_pair, interval=interval,
                ts_open=ts_open, open_=safe_float(r[3]), high=safe_float(r[2]),
                low=safe_float(r[1]), close=safe_float(r[4]),
                base_volume=safe_float(r[5]), interval_seconds=step, now_ts=now_ts,
            ))
        if docs:
            total += upsert_candles(coll, docs, is_backfill=True)
        cur = (max_row_ts + step) if max_row_ts else (cur + max_per * step)
        time.sleep(delay)
    return total



def _ingest_kraken_recent_ohlc_range(coll, hbot_pair, interval, start_ts, end_ts):
    """Use Kraken's OHLC endpoint for the recent <= 720-bar window."""
    step = INTERVAL_SECONDS[interval]
    now_ts = utc_now_ts()

    bars = fetch_kraken_candles(hbot_pair, interval, since_ts=max(0, start_ts - step))
    if not bars:
        return 0

    docs = []
    for b in bars:
        ts_open = int(b["timestamp"])
        if ts_open < start_ts or ts_open > end_ts:
            continue
        docs.append(normalize_candle_doc(
            connector="kraken", hbot_pair=hbot_pair, interval=interval,
            ts_open=ts_open, open_=b["open"], high=b["high"], low=b["low"],
            close=b["close"], base_volume=b["volume"],
            interval_seconds=step, now_ts=now_ts,
        ))
    if docs:
        return upsert_candles(coll, docs, is_backfill=True)
    return 0


def _ingest_kraken_trades_range(coll, hbot_pair, interval, start_ts, end_ts):
    """Deep-history Kraken backfill via /0/public/Trades + local aggregation.

    This scans trades forward from `start_ts`, aggregates them into candles for
    `interval`, and writes only buckets whose open timestamp falls inside the
    requested range [start_ts, end_ts].

    Important:
    - This is much heavier than Kraken's OHLC endpoint.
    - It is the only public REST path Kraken documents for deep history.
    - Missing candles inside a successfully scanned range can be legitimate
      "no trades happened" intervals; Cell 11 records coverage receipts so
      those ranges are not retried forever.
    """
    ex_cfg = EXCHANGES["kraken"]
    if ex_cfg["interval_map"].get(interval) is None:
        return 0

    step = INTERVAL_SECONDS[interval]
    start_ts = align_floor(start_ts, step)
    end_ts = align_floor(end_ts, step)
    if end_ts < start_ts:
        return 0

    now_ts = utc_now_ts()
    cursor = str(max(0, int(start_ts) * 1_000_000_000 - 1))
    total_written = 0
    pending_docs = []
    current = None
    page = 0
    progress_every = 100
    last_cursor = None
    no_progress = 0
    reached_end = False

    def flush_current():
        nonlocal current, pending_docs, total_written
        if current is None:
            return
        ts_open = current["ts_open"]
        if start_ts <= ts_open <= end_ts:
            pending_docs.append(normalize_candle_doc(
                connector="kraken", hbot_pair=hbot_pair, interval=interval,
                ts_open=ts_open,
                open_=current["open"], high=current["high"],
                low=current["low"], close=current["close"],
                base_volume=current["base_volume"],
                quote_volume=current["quote_volume"],
                trade_count=current["trade_count"],
                interval_seconds=step, now_ts=now_ts,
            ))
            if len(pending_docs) >= 2000:
                total_written += upsert_candles(coll, pending_docs, is_backfill=True)
                pending_docs = []
        current = None

    while True:
        trades, next_cursor = fetch_kraken_trades(hbot_pair, since_ns=cursor)
        page += 1

        if not trades:
            break

        for tr in trades:
            ts_float = float(tr["timestamp"])
            bucket_open = align_floor(int(ts_float), step)

            if bucket_open < start_ts:
                continue
            if bucket_open > end_ts:
                reached_end = True
                break

            price = tr["price"]
            volume = tr["volume"]

            if current is None:
                current = {
                    "ts_open": bucket_open,
                    "open": price,
                    "high": price,
                    "low": price,
                    "close": price,
                    "base_volume": volume,
                    "quote_volume": price * volume,
                    "trade_count": 1,
                }
                continue

            if bucket_open != current["ts_open"]:
                flush_current()
                current = {
                    "ts_open": bucket_open,
                    "open": price,
                    "high": price,
                    "low": price,
                    "close": price,
                    "base_volume": volume,
                    "quote_volume": price * volume,
                    "trade_count": 1,
                }
            else:
                current["high"] = max(current["high"], price)
                current["low"] = min(current["low"], price)
                current["close"] = price
                current["base_volume"] += volume
                current["quote_volume"] += price * volume
                current["trade_count"] += 1

        if reached_end:
            break

        next_cursor = str(next_cursor) if next_cursor is not None else str(cursor)
        if next_cursor == str(cursor) or next_cursor == str(last_cursor):
            no_progress += 1
        else:
            no_progress = 0

        if no_progress >= 2:
            raise RuntimeError(
                f"[kraken] {hbot_pair} {interval}: cursor stopped advancing at {cursor}"
            )

        if page % progress_every == 0:
            try:
                through_ts = int(float(trades[-1]["timestamp"]))
                print(f"  [kraken] {hbot_pair} {interval}: page {page:,}, scanned through {fmt_ts(through_ts)}")
            except Exception:
                pass

        last_cursor = cursor
        cursor = next_cursor

    flush_current()
    if pending_docs:
        total_written += upsert_candles(coll, pending_docs, is_backfill=True)
    return total_written


def ingest_kraken_range(coll, hbot_pair, interval, start_ts, end_ts):
    """Hybrid Kraken ingester.

    - Deep history: /0/public/Trades + local aggregation.
    - Recent <= 720 bars: /0/public/OHLC for efficiency.
    """
    ex_cfg = EXCHANGES["kraken"]
    if ex_cfg["interval_map"].get(interval) is None:
        return 0

    step = INTERVAL_SECONDS[interval]
    start_ts = align_floor(start_ts, step)
    end_ts = align_floor(end_ts, step)
    if end_ts < start_ts:
        return 0

    now_ts = utc_now_ts()
    recent_floor = align_floor(now_ts - (ex_cfg["max_per_request"] * step), step)

    total = 0
    if start_ts < recent_floor:
        deep_end = min(end_ts, recent_floor - step)
        if deep_end >= start_ts:
            total += _ingest_kraken_trades_range(coll, hbot_pair, interval, start_ts, deep_end)

    if end_ts >= recent_floor:
        recent_start = max(start_ts, recent_floor)
        if end_ts >= recent_start:
            total += _ingest_kraken_recent_ohlc_range(coll, hbot_pair, interval, recent_start, end_ts)

    return total

INGESTERS = {
    "nonkyc": ingest_nonkyc_range,
    "binance": ingest_binance_range,
    "mexc": ingest_mexc_range,
    "coinbase": ingest_coinbase_range,
    "kraken": ingest_kraken_range,
}

# Sanity check: every configured exchange must have a fetcher + ingester.
_missing = []
for ex in EXCHANGES.keys():
    if ex not in FETCHERS:
        _missing.append(f"fetcher for {ex}")
    if ex not in INGESTERS:
        _missing.append(f"ingester for {ex}")
if _missing:
    print(f"⚠ Missing implementations (edit Cells 3 and 4): {', '.join(_missing)}")
else:
    print(f"Ingesters loaded: {', '.join(sorted(INGESTERS.keys()))}")


Ingesters loaded: binance, coinbase, kraken, mexc, nonkyc


In [5]:

# ── Cell 5: Connect to MongoDB & ensure indexes ─────────────────────────────

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10000)
client.admin.command("ping")
db = client[MONGO_DATABASE]
coll = db[COLLECTION_NAME]
probe_coll = db[PROBE_COLL_NAME]
pair_start_coll = db[PAIR_START_COLL_NAME]
coverage_coll = db[COVERAGE_COLL_NAME]

coll.create_index(
    [("connector", 1), ("trading_pair", 1), ("interval", 1), ("timestamp", 1)],
    unique=True, name="idx_connector_pair_interval_ts",
)
probe_coll.create_index(
    [("exchange", 1), ("interval", 1)],
    unique=True, name="idx_exchange_interval",
)
pair_start_coll.create_index(
    [("exchange", 1), ("pair", 1)],
    unique=True, name="idx_exchange_pair_start",
)
coverage_coll.create_index(
    [("connector", 1), ("trading_pair", 1), ("interval", 1), ("start_ts", 1), ("end_ts", 1)],
    unique=True, name="idx_series_coverage_exact",
)

print(f"Connected to MongoDB: {MONGO_URI.split('@')[-1]}")
print(f"Database: {MONGO_DATABASE}  Candles collection: {COLLECTION_NAME}")
print(f"Probe cache: {PROBE_COLL_NAME}  Pair-start cache: {PAIR_START_COLL_NAME}")
print(f"Coverage receipts: {COVERAGE_COLL_NAME}")
print(f"Total candle documents: {coll.estimated_document_count():,}")
print(f"Cached probes: {probe_coll.estimated_document_count()}")
print(f"Cached pair-starts: {pair_start_coll.estimated_document_count()}")
print(f"Coverage receipts: {coverage_coll.estimated_document_count()}")


Connected to MongoDB: 192.168.1.54:27017/quants_lab?authSource=admin&retryWrites=true&w=majority
Database: quants_lab  Candles collection: candles
Probe cache: candle_coverage_probes  Pair-start cache: candle_coverage_probes_pair_starts
Coverage receipts: candles_coverage
Total candle documents: 25,576,156
Cached probes: 37
Cached pair-starts: 0
Coverage receipts: 0


In [6]:

# ── Cell 6: Earliest-available probe (one-time per exchange × interval) ─────
#
# For exchanges with proper historical candle pagination, we probe an
# exchange-wide earliest timestamp per interval and cache it in Mongo.
#
# Kraken is different:
#   - its OHLC endpoint only returns the most recent 720 bars, regardless of
#     the `since` value, so an exchange-wide OHLC probe is not meaningful;
#   - its public Trades endpoint can start from the beginning of a market
#     (`since=0`), so we cache pair-specific market starts instead.

REFERENCE_PAIR = {
    "nonkyc":   "BTC-USDT",
    "binance":  "BTC-USDT",
    "mexc":     "BTC-USDT",
    "coinbase": "BTC-USD",
    "kraken":   "BTC-USD",
}


def get_cached_pair_start_ts(exchange: str, hbot_pair: str):
    doc = pair_start_coll.find_one(
        {"exchange": exchange, "pair": hbot_pair},
        projection={"pair_start_ts": 1, "_id": 0},
    )
    if not doc:
        return None
    ts = doc.get("pair_start_ts")
    return int(ts) if ts is not None else None


def cache_pair_start_ts(exchange: str, hbot_pair: str, pair_start_ts: int, source: str = "probe"):
    now_ts = utc_now_ts()
    pair_start_coll.update_one(
        {"exchange": exchange, "pair": hbot_pair},
        {
            "$set": {
                "exchange": exchange,
                "pair": hbot_pair,
                "pair_start_ts": int(pair_start_ts),
                "source": source,
                "updated_at": now_ts,
            },
            "$setOnInsert": {"created_at": now_ts},
        },
        upsert=True,
    )


def probe_kraken_pair_start_ts(hbot_pair: str, force: bool = False):
    cached = get_cached_pair_start_ts("kraken", hbot_pair)
    if cached is not None and not force:
        return cached

    trades, _cursor = fetch_kraken_trades(hbot_pair, since_ns=0)
    if not trades:
        return None

    first_trade_ts = int(math.floor(float(trades[0]["timestamp"])))
    cache_pair_start_ts("kraken", hbot_pair, first_trade_ts, source="trades_since_zero")
    return first_trade_ts


def _probe_has_data(exchange: str, interval: str, center_ts: int) -> bool:
    """Return True iff the exchange serves any candles in a small window
    centered on `center_ts`. Uses the reference pair for that exchange."""
    ex_cfg = EXCHANGES[exchange]
    if ex_cfg["interval_map"].get(interval) is None:
        return False
    ref = REFERENCE_PAIR.get(exchange)
    if ref is None:
        raise RuntimeError(f"No REFERENCE_PAIR configured for {exchange}; add one at the top of Cell 6.")
    step = INTERVAL_SECONDS[interval]
    half = 5 * step
    start_ts = max(0, center_ts - half)
    end_ts = center_ts + half

    try:
        if exchange == "nonkyc":
            bars = fetch_nonkyc_candles(ref, interval, to_ts=end_ts, count=10)
            return any(start_ts <= int(b["timestamp"]) <= end_ts for b in bars)
        elif exchange == "binance":
            rows = fetch_binance_candles(ref, interval, start_ts * 1000, end_ts * 1000, 10)
            return len(rows) > 0
        elif exchange == "mexc":
            rows = fetch_mexc_candles(ref, interval, start_ts * 1000, end_ts * 1000, 10)
            return len(rows) > 0
        elif exchange == "coinbase":
            rows = fetch_coinbase_candles(ref, interval, start_ts, end_ts)
            return len(rows) > 0
        elif exchange == "kraken":
            # Only meaningful near the present; Kraken's deep-history floor is
            # pair-specific and handled via Trades rather than this probe.
            bars = fetch_kraken_candles(ref, interval, since_ts=max(0, start_ts - step))
            return any(start_ts <= int(b["timestamp"]) <= end_ts for b in bars)
        else:
            raise RuntimeError(f"No probe branch implemented for {exchange}")
    except Exception as e:
        print(f"  probe error {exchange} {interval} @ {fmt_ts(center_ts)}: {e}")
        return False


def probe_earliest_available_ts(exchange: str, interval: str, absolute_max_days: int, force: bool = False) -> int:
    """Return the earliest-available UTC timestamp (seconds) for a given
    (exchange, interval), using binary search. Cached in Mongo.

    Kraken is special-cased: its actual market start is pair-specific, so this
    function only returns the absolute ceiling for Kraken and Cell 8 later
    refines each pair via `probe_kraken_pair_start_ts`.
    """
    now_ts = utc_now_ts()

    if exchange == "kraken":
        return now_ts - absolute_max_days * 86400

    cached = probe_coll.find_one({"exchange": exchange, "interval": interval})
    if cached and not force:
        age_days = (now_ts - cached["earliest_ts"]) / 86400
        if age_days <= absolute_max_days:
            return int(cached["earliest_ts"])

    lo = now_ts - absolute_max_days * 86400
    hi = now_ts - INTERVAL_SECONDS[interval]

    if not _probe_has_data(exchange, interval, center_ts=hi):
        probe_coll.update_one(
            {"exchange": exchange, "interval": interval},
            {"$set": {"exchange": exchange, "interval": interval,
                      "earliest_ts": now_ts, "no_data": True, "probed_at": now_ts}},
            upsert=True,
        )
        return now_ts

    if _probe_has_data(exchange, interval, center_ts=lo):
        earliest = lo
    else:
        step = INTERVAL_SECONDS[interval]
        low, high = lo, hi
        tolerance = max(2 * 86400, 10 * step)
        while high - low > tolerance:
            mid = (low + high) // 2
            if _probe_has_data(exchange, interval, center_ts=mid):
                high = mid
            else:
                low = mid
        earliest = high

    probe_coll.update_one(
        {"exchange": exchange, "interval": interval},
        {"$set": {"exchange": exchange, "interval": interval,
                  "earliest_ts": earliest, "no_data": False, "probed_at": now_ts}},
        upsert=True,
    )
    return earliest


print("Probe helpers loaded. Kraken pair-start helpers are ready too.")


Probe helpers loaded. Kraken pair-start helpers are ready too.


In [7]:

# ── Cell 7: Run probes for every configured exchange × interval ─────────────
#
# Builds the effective per-(exchange, interval) window for discovery. Kraken is
# special-cased: Cell 7 does NOT use OHLC to limit Kraken because Kraken's OHLC
# endpoint only exposes the most recent 720 bars. Pair-specific starts are
# discovered/cached later in Cell 8 via the Trades endpoint.

FORCE_REPROBE = False

now_ts = utc_now_ts()
absolute_floor_ts = now_ts - ABSOLUTE_MAX_DAYS * 86400

effective_windows = {}   # (exchange, interval) -> earliest_ts

print("Probing each (exchange, interval) — this is cached after first run.")
print(f"Absolute ceiling: {ABSOLUTE_MAX_DAYS} days  (earliest date: {fmt_ts(absolute_floor_ts)})\n")
print(f"{'Exchange':12s} {'Interval':8s} {'Target days':>12s} {'Probed ts':>20s} {'Days back':>11s}  Effective start")
print("─" * 95)

for exchange in sorted(EXCHANGES.keys()):
    for interval in sorted(INTERVAL_TARGETS_DAYS.keys(), key=lambda k: INTERVAL_SECONDS[k]):
        if EXCHANGES[exchange]["interval_map"].get(interval) is None:
            continue

        target_days = min(INTERVAL_TARGETS_DAYS[interval], ABSOLUTE_MAX_DAYS)
        target_floor_ts = now_ts - target_days * 86400

        if exchange == "kraken":
            effective_start = target_floor_ts
            effective_windows[(exchange, interval)] = effective_start
            print(f"{exchange:12s} {interval:8s} {target_days:>12d} {'pair-specific':>20s} "
                  f"{'n/a':>10s}  {fmt_ts(effective_start)}")
            continue

        probed_floor_ts = probe_earliest_available_ts(exchange, interval, ABSOLUTE_MAX_DAYS, force=FORCE_REPROBE)
        effective_start = max(target_floor_ts, probed_floor_ts)
        effective_windows[(exchange, interval)] = effective_start

        days_back = (now_ts - probed_floor_ts) / 86400
        print(f"{exchange:12s} {interval:8s} {target_days:>12d} {fmt_ts(probed_floor_ts):>20s} "
              f"{days_back:>10.1f}  {fmt_ts(effective_start)}")

print("\nProbes complete. effective_windows is ready.")
print("Kraken pair starts are discovered lazily in Cell 8 and cached in Mongo.")


Probing each (exchange, interval) — this is cached after first run.
Absolute ceiling: 3650 days  (earliest date: 2016-04-25 03:50)

Exchange     Interval  Target days            Probed ts   Days back  Effective start
───────────────────────────────────────────────────────────────────────────────────────────────
binance      1m                365     2026-04-17 02:51        6.0  2026-04-17 02:51
binance      5m               1825     2026-04-17 02:51        6.0  2026-04-17 02:51
binance      15m              1825     2026-04-17 02:51        6.0  2026-04-17 02:51
binance      30m              1825     2026-04-17 02:51        6.0  2026-04-17 02:51
binance      1h               3650     2026-04-17 02:51        6.0  2026-04-17 02:51
binance      4h               3650     2026-04-17 02:51        6.0  2026-04-17 02:51
binance      1d               3650     2026-04-17 02:51        6.0  2026-04-17 02:51
coinbase     1m                365     2026-04-20 13:23        2.6  2026-04-20 13:23
coinbas

In [8]:

# ── Cell 8: Discover all (connector, pair, interval) combos and show gaps ──
#
# In addition to the gap/uncovered summary, this cell now prints the date
# ranges that are already covered for each series.
#
# For Kraken:
#   - pair starts are discovered once per pair via Trades `since=0`;
#   - coverage receipts are unioned with actual candles so that intervals with
#     no trades do not look like permanent gaps after a successful scan.

DISCOVER_KRAKEN_PAIR_STARTS = True
COVERAGE_RANGE_PREVIEW_LIMIT = 5

pipeline = [
    {"$group": {
        "_id": {"connector": "$connector", "trading_pair": "$trading_pair", "interval": "$interval"},
        "count": {"$sum": 1},
        "min_ts": {"$min": "$timestamp"},
        "max_ts": {"$max": "$timestamp"},
    }},
    {"$sort": {"_id.connector": 1, "_id.trading_pair": 1, "_id.interval": 1}},
]
raw_combos = list(coll.aggregate(pipeline, allowDiskUse=True))

combos = []
kraken_pair_start_cache = {}

for doc in raw_combos:
    connector = doc["_id"]["connector"]
    pair = doc["_id"]["trading_pair"]
    interval = doc["_id"]["interval"]

    if connector not in EXCHANGES:
        continue
    if EXCHANGES[connector]["interval_map"].get(interval) is None:
        continue

    step = INTERVAL_SECONDS.get(interval)
    if step is None:
        continue

    window_start = effective_windows.get((connector, interval))
    if window_start is None:
        continue

    pair_start_ts = None
    if connector == "kraken" and DISCOVER_KRAKEN_PAIR_STARTS:
        if pair not in kraken_pair_start_cache:
            try:
                kraken_pair_start_cache[pair] = probe_kraken_pair_start_ts(pair)
            except Exception as e:
                print(f"  [kraken] pair-start probe failed for {pair}: {e}")
                kraken_pair_start_cache[pair] = None
        pair_start_ts = kraken_pair_start_cache[pair]
        if pair_start_ts is not None:
            window_start = align_floor(pair_start_ts, step)

    effective_start = align_floor(window_start, step)

    recent_cutoff = now_ts - (RECENT_DAYS_OWNED * 86400) if RECENT_DAYS_OWNED > 0 else now_ts
    effective_end = min(
        last_closed_open_ts(now_ts, step),
        last_closed_open_ts(recent_cutoff, step),
    )

    summary = summarize_series_window(
        coll, connector, pair, interval, step, effective_start, effective_end
    )

    combos.append({
        "connector": connector,
        "pair": pair,
        "interval": interval,
        "total_docs": int(doc["count"]),
        "docs_in_window": summary["docs_in_window"],
        "covered_buckets": summary["covered_buckets"],
        "expected": summary["expected"],
        "missing": summary["missing"],
        "pct": summary["pct"],
        "step": step,
        "window_start": effective_start,
        "window_end": summary["effective_end"],
        "covered_ranges": summary["covered_ranges"],
        "covered_range_display": format_ts_ranges(summary["covered_ranges"], limit=COVERAGE_RANGE_PREVIEW_LIMIT),
        "doc_ranges": summary["doc_ranges"],
        "recorded_ranges": summary["recorded_ranges"],
        "first_doc_ts": doc.get("min_ts"),
        "last_doc_ts": doc.get("max_ts"),
        "pair_start_ts": pair_start_ts,
    })

connectors_found = sorted(set(c["connector"] for c in combos))
print(f"Found {len(combos)} (connector, pair, interval) combos already in Mongo "
      f"across {len(connectors_found)} connector(s)")

print("\nConnector Summary:")
for conn in connectors_found:
    conn_combos = [c for c in combos if c["connector"] == conn]
    total_missing = sum(c["missing"] for c in conn_combos)
    pairs = len(set(c["pair"] for c in conn_combos))
    intervals = len(set(c["interval"] for c in conn_combos))
    row_indices = [i for i, c in enumerate(combos) if c["connector"] == conn]
    print(f"  {conn:12s} — {pairs} pairs × {intervals} intervals = {len(conn_combos)} series, "
          f"{total_missing:,} uncovered intervals  (rows {row_indices[0]}–{row_indices[-1]})")

print(f"\n{'─' * 128}")
print(f"{'Idx':>5s}  {'Connector':12s} {'Pair':16s} {'Intv':6s} {'Docs':>10s} {'Covered':>10s} {'Expected':>10s} {'Missing':>10s} {'Complete':>9s}")
print(f"{'─' * 128}")
for i, c in enumerate(combos):
    flag = " ✓" if c["missing"] == 0 else f" ← {c['missing']:,}"
    print(f"{i:>5d}  {c['connector']:12s} {c['pair']:16s} {c['interval']:6s} "
          f"{c['docs_in_window']:>10,d} {c['covered_buckets']:>10,d} "
          f"{c['expected']:>10,d} {c['missing']:>10,d} {c['pct']:>8.1f}%{flag}")
print(f"{'─' * 128}")

print("\nCovered date ranges per series (within the active backfill window):")
for i, c in enumerate(combos):
    extra = ""
    if c["connector"] == "kraken" and c.get("pair_start_ts") is not None:
        extra = f"  [pair start {fmt_ts(c['pair_start_ts'])}]"
    print(f"  [{i:>4d}] {c['connector']:12s} {c['pair']:16s} {c['interval']:6s}  "
          f"{c['covered_range_display']}{extra}")


  [kraken] pair-start probe failed for BTC-USD: HTTPSConnectionPool(host='api.kraken.com', port=443): Max retries exceeded with url: /0/public/Trades?pair=XBTUSD&since=0 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x774b1c7eb3b0>: Failed to establish a new connection: [Errno -2] Name or service not known'))
  [kraken] pair-start probe failed for ETH-USD: HTTPSConnectionPool(host='api.kraken.com', port=443): Max retries exceeded with url: /0/public/Trades?pair=ETHUSD&since=0 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x774b1c7eb350>: Failed to establish a new connection: [Errno -2] Name or service not known'))
  [kraken] pair-start probe failed for SOL-USD: HTTPSConnectionPool(host='api.kraken.com', port=443): Max retries exceeded with url: /0/public/Trades?pair=SOLUSD&since=0 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x774b1c7eb6e0>: Failed to establish a new connection: [Errno -2] Nam

In [9]:

# ── Cell 9: Select + preview ────────────────────────────────────────────────
#
# Selection modes:
#   SELECTION = "all"               — every combo with uncovered ranges
#   SELECTION = "binance"           — all series for one connector
#   SELECTION = ["binance", "mexc"] — all series for multiple connectors
#   SELECTION = [0, 1, 5]           — specific row indices from Cell 8
#
# NEW_PAIRS: if you want to backfill a pair that is not yet in Mongo at all,
# list it here as (connector, pair) tuples. The script will fetch full history
# within the configured windows. Kraken NEW_PAIRS use the cached/probed pair
# start when available.

SELECTION = "all"
NEW_PAIRS = []   # e.g. [("binance", "SOL-USDT"), ("kraken", "XMR-USD")]

if SELECTION == "all":
    selected_combos = [c for c in combos if c["missing"] > 0]
elif isinstance(SELECTION, str):
    selected_combos = [c for c in combos if c["connector"] == SELECTION and c["missing"] > 0]
elif isinstance(SELECTION, (list, tuple)) and all(isinstance(x, str) for x in SELECTION):
    selected_combos = [c for c in combos if c["connector"] in set(SELECTION) and c["missing"] > 0]
else:
    indices = list(SELECTION)
    selected_combos = [combos[i] for i in indices if i < len(combos) and combos[i]["missing"] > 0]

for (nconn, npair) in NEW_PAIRS:
    if nconn not in EXCHANGES:
        print(f"  ⚠ NEW_PAIRS skipping {nconn}:{npair} — exchange not in config")
        continue

    cached_pair_start = None
    if nconn == "kraken":
        try:
            cached_pair_start = probe_kraken_pair_start_ts(npair)
        except Exception as e:
            print(f"  ⚠ NEW_PAIRS {nconn}:{npair} pair-start probe failed: {e}")

    for interval in INTERVAL_TARGETS_DAYS.keys():
        if EXCHANGES[nconn]["interval_map"].get(interval) is None:
            continue

        step = INTERVAL_SECONDS[interval]
        window_start = effective_windows.get((nconn, interval))
        if window_start is None:
            continue

        if cached_pair_start is not None:
            window_start = align_floor(cached_pair_start, step)

        effective_start = align_floor(window_start, step)
        effective_end = last_closed_open_ts(now_ts, step)
        expected = ((effective_end - effective_start) // step) + 1 if effective_end >= effective_start else 0
        if expected == 0:
            continue

        already = any(c["connector"] == nconn and c["pair"] == npair and c["interval"] == interval
                      for c in combos)
        if already:
            continue

        selected_combos.append({
            "connector": nconn,
            "pair": npair,
            "interval": interval,
            "total_docs": 0,
            "docs_in_window": 0,
            "covered_buckets": 0,
            "expected": expected,
            "missing": expected,
            "pct": 0.0,
            "step": step,
            "window_start": effective_start,
            "window_end": effective_end,
            "covered_ranges": [],
            "covered_range_display": "—",
            "pair_start_ts": cached_pair_start,
            "is_new_pair": True,
        })


def estimate_requests(combo):
    ex_cfg = EXCHANGES[combo["connector"]]
    max_per = ex_cfg["max_per_request"]
    if combo["connector"] == "kraken":
        # Lower bound only: deep Kraken backfill may need many Trades pages.
        return max(1, math.ceil(combo["missing"] / max_per))
    return max(1, math.ceil(combo["missing"] / max_per))


by_exchange = {}
for c in selected_combos:
    by_exchange.setdefault(c["connector"], []).append(c)

print(f"Selection: {len(selected_combos)} series to backfill")
print(f"{'─' * 92}")
print(f"{'Exchange':12s} {'Series':>8s} {'Uncovered':>12s} {'Est. requests':>15s} {'Est. wall-clock':>18s}")
print(f"{'─' * 92}")
grand_requests = 0
grand_missing = 0
worst_wall_clock = 0.0
for ex in sorted(by_exchange.keys()):
    cs = by_exchange[ex]
    missing = sum(c["missing"] for c in cs)
    reqs = sum(estimate_requests(c) for c in cs)
    delay = EXCHANGES[ex]["request_delay_seconds"]
    secs = reqs * delay
    mins = secs / 60
    grand_requests += reqs
    grand_missing += missing
    worst_wall_clock = max(worst_wall_clock, secs)
    print(f"{ex:12s} {len(cs):>8d} {missing:>12,d} {reqs:>15,d} {mins:>15.1f} min")

print(f"{'─' * 92}")
print(f"{'TOTAL':12s} {len(selected_combos):>8d} {grand_missing:>12,d} {grand_requests:>15,d}  "
      f"(parallel: ~{worst_wall_clock/60:.1f} min)")
print(f"{'─' * 92}")

if "kraken" in by_exchange:
    print("Note: Kraken estimates are lower bounds. Deep-history Kraken backfill uses paginated Trades scans,")
    print("      so actual request count depends on trade volume, not just missing candle buckets.")

if selected_combos:
    print("\nReady. Run Cell 10 to execute.")
else:
    print("\nNothing to backfill.")


Selection: 526 series to backfill
────────────────────────────────────────────────────────────────────────────────────────────
Exchange       Series    Uncovered   Est. requests    Est. wall-clock
────────────────────────────────────────────────────────────────────────────────────────────
kraken             25    5,387,079           7,492             0.0 min
mexc              183    2,766,164           5,660            18.9 min
nonkyc            318    6,812,158           1,592             0.0 min
────────────────────────────────────────────────────────────────────────────────────────────
TOTAL             526   14,965,401          14,744  (parallel: ~18.9 min)
────────────────────────────────────────────────────────────────────────────────────────────
Note: Kraken estimates are lower bounds. Deep-history Kraken backfill uses paginated Trades scans,
      so actual request count depends on trade volume, not just missing candle buckets.

Ready. Run Cell 10 to execute.


In [10]:

# ── Cell 10: Gap detection (streaming, memory-safe) ─────────────────────────

def find_gaps_in_window(coll, connector, pair, interval, step, window_start, window_end):
    effective_start = align_floor(window_start, step)
    effective_end = min(
        align_floor(window_end, step),
        last_closed_open_ts(utc_now_ts(), step),
    )
    if effective_end < effective_start:
        return []
    summary = summarize_series_window(
        coll, connector, pair, interval, step, effective_start, effective_end
    )
    return summary["gap_ranges"]


print("Gap detection loaded.")


Gap detection loaded.


In [ ]:

# Cell 11: Run the backfill
#
# Layered parallelism:
#   - Outer: one thread per exchange.
#   - Inner: per-exchange parallel pair-workers (from _EXCHANGE_TUNING).
#   - Pacing: shared per-exchange token bucket enforces sustained rate caps.
# All pair-workers within an exchange share the same bucket, so adding
# workers raises parallelism without exceeding the rate-per-second cap.

if not selected_combos:
    print("Nothing to backfill.")
else:
    per_exchange = {}
    for c in selected_combos:
        per_exchange.setdefault(c["connector"], []).append(c)

    results = {}
    results_lock = threading.Lock()
    counter_lock = threading.Lock()
    total_series = len(selected_combos)
    counter = {"done": 0}

    def exchange_worker(exchange: str, combos_for_ex: list):
        ingester = INGESTERS.get(exchange)
        if ingester is None:
            with results_lock:
                results[exchange] = {"written": 0,
                                     "errors": [f"{exchange}: no ingester"],
                                     "gap_ranges": 0}
            return

        parallel_workers = get_exchange_parallel_workers(exchange)
        rate_cap = _EXCHANGE_TUNING.get(exchange, {}).get("rate_per_sec", "?")
        print(f"[{exchange}] {len(combos_for_ex)} series, "
              f"{parallel_workers} parallel pair-workers, rate cap = {rate_cap} req/s")

        local_written = 0
        local_errors: list = []
        local_gaps = 0
        local_lock = threading.Lock()

        def one_combo(combo):
            combo_started = time.perf_counter()
            combo_written = 0
            pair = combo["pair"]
            interval = combo["interval"]
            step = combo["step"]
            is_new = combo.get("is_new_pair", False)
            win_start = combo["window_start"]
            win_end = combo.get("window_end", last_closed_open_ts(utc_now_ts(), step))

            if is_new:
                eff_start = align_floor(win_start, step)
                eff_end = win_end
                gaps = [(eff_start, eff_end)] if eff_end >= eff_start else []
            else:
                gaps = find_gaps_in_window(coll, exchange, pair, interval, step,
                                            win_start, win_end)

            if not gaps:
                with counter_lock:
                    counter["done"] += 1
                    done = counter["done"]
                sweep_elapsed = time.time() - start_wall
                eta = (sweep_elapsed / done * (total_series - done)) if done > 0 else 0
                print(f"[{done}/{total_series}] {exchange} {pair} {interval} "
                      f"✓ no uncovered ranges  [sweep {sweep_elapsed/60:.1f}min, ETA {eta/60:.1f}min]")
                return 0, 0, []

            errors_here = []
            for gs, ge in gaps:
                try:
                    combo_written += ingester(coll, pair, interval, gs, ge)
                    if exchange == "kraken":
                        record_coverage_range(
                            coverage_coll, exchange, pair, interval, gs, ge,
                            source="backfill_success",
                        )
                except Exception as e:
                    err = f"{exchange} {pair} {interval} [{fmt_ts(gs)}→{fmt_ts(ge)}]: {e}"
                    errors_here.append(err)
                    print(f"  ✗ {err}")

            combo_elapsed = time.perf_counter() - combo_started
            with counter_lock:
                counter["done"] += 1
                done = counter["done"]
            sweep_elapsed = time.time() - start_wall
            eta = (sweep_elapsed / done * (total_series - done)) if done > 0 else 0
            print(f"[{done}/{total_series}] {exchange} {pair} {interval}: "
                  f"wrote {combo_written:,} across {len(gaps)} range(s) "
                  f"in {combo_elapsed:.1f}s  "
                  f"[sweep {sweep_elapsed/60:.1f}min, ETA {eta/60:.1f}min]")
            return combo_written, len(gaps), errors_here

        with ThreadPoolExecutor(max_workers=parallel_workers) as pair_pool:
            futs = {pair_pool.submit(one_combo, c): c for c in combos_for_ex}
            for fut in as_completed(futs):
                try:
                    w, g, errs = fut.result()
                    with local_lock:
                        local_written += w
                        local_gaps += g
                        local_errors.extend(errs)
                except Exception as e:
                    c = futs[fut]
                    with local_lock:
                        local_errors.append(
                            f"{exchange} {c['pair']} {c['interval']}: worker crashed: {e}"
                        )

        with results_lock:
            results[exchange] = {"written": local_written,
                                 "errors": local_errors,
                                 "gap_ranges": local_gaps}

    start_wall = time.time()
    n_exchange_workers = max(1, len(per_exchange))
    print(f"Running {n_exchange_workers} exchange(s) in parallel, "
          f"with per-exchange parallel pair-workers as configured.\n")

    with ThreadPoolExecutor(max_workers=n_exchange_workers) as pool:
        futures = {pool.submit(exchange_worker, ex, cs): ex
                   for ex, cs in per_exchange.items()}
        for fut in as_completed(futures):
            ex = futures[fut]
            try:
                fut.result()
            except Exception as e:
                print(f"  ✗ Worker for {ex} crashed: {e}")

    elapsed = time.time() - start_wall

    print(f"\n{'═' * 70}")
    print("BACKFILL COMPLETE")
    print(f"  Wall clock: {elapsed/60:.1f} min")
    total_written = sum(r["written"] for r in results.values())
    total_errors = sum(len(r["errors"]) for r in results.values())
    total_gaps = sum(r["gap_ranges"] for r in results.values())
    print(f"  Gap / uncovered ranges processed: {total_gaps:,}")
    print(f"  Total candles written: {total_written:,}")
    print(f"  Errors: {total_errors}")
    for ex in sorted(results.keys()):
        r = results[ex]
        print(f"    {ex:12s} wrote {r['written']:>10,d} across "
              f"{r['gap_ranges']:>4} ranges, {len(r['errors'])} error(s)")
        try:
            s = _get_exchange_bucket(ex).stats()
            initial = _EXCHANGE_TUNING.get(ex, {}).get("rate_per_sec", "?")
            print(f"      final bucket rate: {s['rate_per_sec']:.2f} req/s "
                  f"(initial: {initial})")
            if isinstance(initial, (int, float)) and s["rate_per_sec"] < initial:
                print(f"      ↪ rate was auto-decayed — {ex} returned 429s during the run")
        except Exception:
            pass
        for e in r["errors"][:5]:
            print(f"      ✗ {e}")
        if len(r["errors"]) > 5:
            print(f"      ... and {len(r['errors']) - 5} more")
    print(f"{'═' * 70}")


Running 2 exchange(s) in parallel, with per-exchange parallel pair-workers as configured.

[mexc] 189 series, 2 parallel pair-workers, rate cap = 5.0 req/s
[nonkyc] 332 series, 8 parallel pair-workers, rate cap = 4.0 req/s
[1/521] nonkyc AAVE-USDT 4h: wrote 0 across 1 range(s) in 0.6s  [sweep 0.0min, ETA 5.3min]
[2/521] nonkyc AAVE-USDT 1d: wrote 0 across 1 range(s) in 1.0s  [sweep 0.0min, ETA 4.5min]
[3/521] nonkyc AAVE-USDT 12h: wrote 0 across 1 range(s) in 1.1s  [sweep 0.0min, ETA 3.1min]
[4/521] nonkyc AAVE-USDT 8h: wrote 0 across 1 range(s) in 1.2s  [sweep 0.0min, ETA 2.5min]
[5/521] nonkyc ADA-USDT 12h: wrote 4 across 2 range(s) in 1.8s  [sweep 0.0min, ETA 3.2min]
[6/521] mexc ADA-USDT 15m: wrote 287 across 2 range(s) in 2.5s  [sweep 0.0min, ETA 3.5min]
[7/521] mexc ADA-USDT 1d: wrote 3 across 2 range(s) in 2.7s  [sweep 0.0min, ETA 3.3min]
[8/521] nonkyc ADA-USDT 1d: wrote 2 across 2 range(s) in 1.9s  [sweep 0.0min, ETA 3.1min]
[9/521] nonkyc ADA-USDT 1h: wrote 58 across 2 range(

In [ ]:

# ── Cell 12: Post-backfill verification ──────────────────────────────────────

selected_connectors_set = set(c["connector"] for c in selected_combos)

print(f"Post-backfill verification ({', '.join(sorted(selected_connectors_set))})")
print(f"{'─' * 128}")
print(f"{'Idx':>5s}  {'Connector':12s} {'Pair':16s} {'Intv':6s} {'Docs':>10s} {'Covered':>10s} {'Expected':>10s} {'Missing':>10s} {'Complete':>9s}")
print(f"{'─' * 128}")

still_missing_total = 0

pipeline = [
    {"$group": {
        "_id": {"connector": "$connector", "trading_pair": "$trading_pair", "interval": "$interval"},
        "count": {"$sum": 1},
    }},
    {"$sort": {"_id.connector": 1, "_id.trading_pair": 1, "_id.interval": 1}},
]
all_combos_after = list(coll.aggregate(pipeline, allowDiskUse=True))

for i, agg in enumerate(all_combos_after):
    connector = agg["_id"]["connector"]
    if connector not in selected_connectors_set:
        continue

    pair = agg["_id"]["trading_pair"]
    interval = agg["_id"]["interval"]
    if EXCHANGES.get(connector, {}).get("interval_map", {}).get(interval) is None:
        continue

    step = INTERVAL_SECONDS[interval]
    win_start = effective_windows.get((connector, interval))
    if win_start is None:
        continue

    if connector == "kraken":
        cached_pair_start = get_cached_pair_start_ts("kraken", pair)
        if cached_pair_start is not None:
            win_start = align_floor(cached_pair_start, step)

    recent_cutoff = utc_now_ts() - (RECENT_DAYS_OWNED * 86400) if RECENT_DAYS_OWNED > 0 else utc_now_ts()
    win_end = min(
        last_closed_open_ts(utc_now_ts(), step),
        last_closed_open_ts(recent_cutoff, step),
    )

    summary = summarize_series_window(coll, connector, pair, interval, step, win_start, win_end)
    missing = summary["missing"]
    pct = summary["pct"]
    still_missing_total += missing

    flag = " ✓" if missing == 0 else f" ← {missing:,} still uncovered"
    print(f"{i:>5d}  {connector:12s} {pair:16s} {interval:6s} "
          f"{summary['docs_in_window']:>10,d} {summary['covered_buckets']:>10,d} "
          f"{summary['expected']:>10,d} {missing:>10,d} {pct:>8.1f}%{flag}")

print(f"{'─' * 128}")
if still_missing_total == 0:
    print("✓ All selected series are 100% covered!")
else:
    print(f"⚠ {still_missing_total:,} intervals still uncovered "
          f"(or, for Kraken, not yet scanned and recorded as covered)")


In [ ]:
# ── Cell 13 (Optional): Inspect a specific series ──────────────────────────

INSPECT_CONNECTOR = "nonkyc"
INSPECT_PAIR = "XMR-USDT"
INSPECT_INTERVAL = "5m"

step = INTERVAL_SECONDS.get(INSPECT_INTERVAL)
win_start = effective_windows.get((INSPECT_CONNECTOR, INSPECT_INTERVAL))
if step is None or win_start is None:
    print(f"Unknown (exchange, interval): ({INSPECT_CONNECTOR}, {INSPECT_INTERVAL})")
else:
    gaps = find_gaps_in_window(coll, INSPECT_CONNECTOR, INSPECT_PAIR, INSPECT_INTERVAL,
                               step, win_start, utc_now_ts())
    total_missing = sum(((e - s) // step) + 1 for s, e in gaps)
    print(f"Gaps for {INSPECT_CONNECTOR} {INSPECT_PAIR} {INSPECT_INTERVAL}:")
    print(f"  Window start: {fmt_ts(win_start)}")
    print(f"  Total gap ranges: {len(gaps)}")
    print(f"  Total missing candles: {total_missing:,}")
    print()
    for i, (gs, ge) in enumerate(gaps[:20]):
        cnt = ((ge - gs) // step) + 1
        dur_h = cnt * step / 3600
        print(f"  [{i+1:3d}] {fmt_ts(gs)} → {fmt_ts(ge)}  ({cnt:>6,d} candles, {dur_h:>7.1f}h)")
    if len(gaps) > 20:
        print(f"  ... and {len(gaps) - 20} more ranges")


## Smoke test for NonKYC pacing

Delete this cell + the code cell below after one successful run.
The code cell issues 20 NonKYC requests and prints the observed throughput.
Expected: 1.5-3.0 seconds for 20 requests (rate cap 12 req/s + ~300ms RTT).
If > 10s, pacing is still serialized somewhere — do NOT run Cell 11 yet.


In [ ]:
# Stress smoke test — validates slow-loris mitigation.
# Expected outcome: 100 requests complete in 15-30 seconds with 2 pair-workers
# (6 req/s cap, plus TCP+TLS handshake per request). No single request should
# exceed the 15s read timeout.
import time as _t
from concurrent.futures import ThreadPoolExecutor, as_completed

N_REQUESTS = 100
N_WORKERS = 2

def _one(i):
    t0 = _t.perf_counter()
    try:
        bars = fetch_nonkyc_candles("BTC-USDT", "1h",
                                    to_ts=int(_t.time()) - i*3600, count=100)
        return ("ok", _t.perf_counter() - t0, len(bars))
    except Exception as e:
        return ("err", _t.perf_counter() - t0, str(e))

_start = _t.perf_counter()
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    results = [f.result() for f in as_completed(
        pool.submit(_one, i) for i in range(N_REQUESTS)
    )]
_wall = _t.perf_counter() - _start

ok = [r for r in results if r[0] == "ok"]
err = [r for r in results if r[0] == "err"]
latencies = sorted(r[1] for r in ok)

print(f"Stress smoke test: {N_REQUESTS} requests, {N_WORKERS} workers")
print(f"  Wall time: {_wall:.2f}s = {N_REQUESTS/_wall:.1f} req/s effective")
print(f"  OK: {len(ok)}  Errors: {len(err)}")
if latencies:
    p50 = latencies[len(latencies)//2]
    p95 = latencies[int(len(latencies)*0.95)]
    p99 = latencies[int(len(latencies)*0.99)]
    pmax = latencies[-1]
    print(f"  Latency p50={p50*1000:.0f}ms  p95={p95*1000:.0f}ms  "
          f"p99={p99*1000:.0f}ms  max={pmax*1000:.0f}ms")
print(f"  Bucket stats: {_get_exchange_bucket('nonkyc').stats()}")
print()
if len(err) > 0:
    print(f"  ⚠  Errors occurred:")
    for e in err[:5]:
        print(f"      {e[2]}")
    print(f"      (showing first 5 of {len(err)})")
if latencies and latencies[-1] > 14.5:
    print(f"  ✗ FAIL — at least one request hung near the 15s timeout.")
    print(f"    Slow-loris is still occurring. Do NOT run full backfill.")
elif len(err) > len(ok) * 0.05:
    print(f"  ⚠  >5% errors. Consider lowering rate_per_sec or parallel.")
elif N_REQUESTS / _wall < 4.0:
    print(f"  ⚠  Throughput below 4 req/s — slower than expected with 2 workers.")
    print(f"    May still be fine; try full backfill but watch for stalls.")
else:
    print(f"  ✓ PASS — no stalls, no errors, sustained throughput looks healthy.")
    print(f"    Ready to run Cell 11 for full backfill.")

In [ ]:
# Run the EXACT same ingester function the notebook uses, but serially and
# observably. Just the first few pairs. This isolates whether the hang is in
# the ingester itself or in Cell 11's threading wrapper.

import time

# Force a clean bucket state — reset it so we don't inherit any weirdness
_exchange_buckets.pop("nonkyc", None)

test_cases = [
    ("AAVE-USDT", "12h"),  # pair 1 — writes 0
    ("AAVE-USDT", "15m"),  # pair 2
    ("AAVE-USDT", "1d"),   # pair 3
    ("AAVE-USDT", "1h"),   # pair 4
    ("AAVE-USDT", "4h"),   # pair 5
    ("AAVE-USDT", "8h"),   # pair 6 — the one that hangs
    ("AAVE-USDT", "5m"),   # pair 7
]

for i, (pair, interval) in enumerate(test_cases, 1):
    step = INTERVAL_SECONDS[interval]
    target_days = CFG["interval_targets_days"][interval]
    target_days = min(target_days, CFG["absolute_max_days"])
    end_ts = utc_now_ts()
    start_ts = end_ts - target_days * 86400

    t0 = time.perf_counter()
    try:
        # Call the SAME ingester Cell 11 calls
        written = ingest_nonkyc_range(coll, pair, interval, start_ts, end_ts)
        dt = time.perf_counter() - t0
        print(f"[{i}] {pair} {interval}: wrote {written} in {dt:.1f}s")
    except Exception as e:
        dt = time.perf_counter() - t0
        print(f"[{i}] {pair} {interval}: FAILED after {dt:.1f}s: {type(e).__name__}: {e}")